# B at D = 128, with and without class weights
The working method (B = `retain_open`: a shadow encoder + decoder keeps what the live encoder must not lose) in the Craftax-shaped grid
world, now with a Craftax-faithful agent and mobs:

* the **agent is an arrow** (its silhouette is the facing, as the sprite is in Craftax), not a white tile with a yellow half;
* the **mob is predictable**: it persists and steps toward the agent with probability 0.7 (else at random), blocked by materials, the border
  and the agent. `twinkle` stays as the only pure noise.

Three runs at D = 128, SIGReg weight 0.01, 15k steps, one seed, the same env:

| run | what it answers |
|---|---|
| **B plain** | does mass x D alone hold the entities (hand 0.88 %: mass x D = 1.1, near the upper bound of kappa_B)? |
| **B + class weights** | what do ground-truth tile-class weights add on top (agent position, the mob, the arrow)? |
| **sg** (no shadow) | the control: if `sg` holds an entity at D = 128 too, B is not what holds it, D is |

The runs go to a **background process**: the notebook only plots, so you can close the tab or restart the kernel and come back. The probe
is an MLP per part at every eval (1000 steps), on the live encoder (solid) and the shadow (dashed). **Read the shadow first**: an entity
missing from the shadow cannot be recovered by retention; present in the shadow and missing live means retention failed.

In [ ]:
%pip install -q -e .
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import display
from gaf.env_grid import GridCfg, component_masses
from gaf.train import Cfg
from gaf import analysis as A

OUT = "/workspace/results_b128"
SEED, D, STEPS, WORKERS = 0, 128, 15_000, 3
ENV = GridCfg(W=12, H=8, hand_tiles=1, agent_shape="arrow", mob_mode="chase", n_mobs=1, mob_chase=0.7)
COMMON = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=STEPS, basis="excess", floor_steps=1500, floor_warm=True,
              eval_every=1000, eval_parts=4096, D=D, seed=SEED, env=ENV)
RUNS = {"B plain": Cfg(arm="retain_open", tag="b128-plain", **COMMON),
        "B + weights": Cfg(arm="retain_open", tag="b128-weights", rec_class_power=1.0, **COMMON),
        "sg": Cfg(arm="sg", tag="b128-sg", **COMMON)}
PARTS = ["hand", "position", "inventory", "facing", "mob", "twinkle"]
pd.set_option("display.precision", 3)

## Masses of the new env (step 0)
Share of the pixel variance of each component. The arrow's facing replaces the marker's (3.0 %); the mob is one tile, as before, but now
predictable.

In [ ]:
m = component_masses(ENV, n=2048)
display(pd.Series({k: f"{v * 100:.2f} %" for k, v in m.items()}, name="share of pixel variance"))

## Launch
Starts the three runs in a background process (3 workers on the GPU). Re-running the cell while they are alive refuses to start twice;
runs that are done are skipped. A run cannot resume: if the pod restarts, the interrupted run starts again.

In [ ]:
pid = A.launch(list(RUNS.values()), OUT, workers=WORKERS)
print("started, pid", pid, "- log:", f"{OUT}/launch.log")

In [ ]:
display(A.progress(OUT, list(RUNS.values())))
print(open(f"{OUT}/launch.log").read()[-600:])

## Live probes
One panel per part, one colour per run, solid = live encoder, dashed = shadow (the `sg` run has none). Redraws every 30 s and stops
when all runs are done; interrupting the cell stops only the plot.

In [ ]:
A.live_probes(OUT, RUNS, every=30)

## Final readout
Live-encoder MLP probe (R² of each part) and the shadow's, at the end of training, then settledness: change over the last 4000 steps and
the swing inside that window (a run that is still moving is provisional).

In [ ]:
df = A.collect(OUT)
rows, shadow, runs = {}, {}, {}
for name, c in RUNS.items():
    r = df[(df.tag == c.tag) & (df.seed == SEED)]
    if not len(r):
        continue
    rows[name] = {**{p: r[f"pm_{p}"].mean() for p in PARTS}, "rank": r["pm_rank"].mean()}
    if f"sh_pm_{PARTS[0]}" in r and r[f"sh_pm_{PARTS[0]}"].notna().any():
        shadow[name] = {p: r[f"sh_pm_{p}"].mean() for p in PARTS}
    runs[name] = dict(arm=c.arm, tag=c.tag, seed=SEED)
print("live encoder"); display(pd.DataFrame(rows).T.round(3))
print("shadow"); display(pd.DataFrame(shadow).T.round(3))

settle = {}
for name, c in RUNS.items():
    try:
        h = pd.read_csv(f"{OUT}/{c.run_id}/metrics.csv")
    except FileNotFoundError:
        continue
    last = h.step.max()
    win = h[h.step >= last - 4000]
    first = h[h.step <= last - 4000].iloc[-1:] if (h.step <= last - 4000).any() else h.iloc[:1]
    settle[name] = {f"{p}": (win[f"ev_pm_{p}"].iloc[-1] - first[f"ev_pm_{p}"].iloc[0], win[f"ev_pm_{p}"].max() - win[f"ev_pm_{p}"].min())
                    for p in PARTS if f"ev_pm_{p}" in h}
print("settled? (change, swing) of the live probe over the last 4000 steps; settled = |change| < 0.05 and swing < 0.1")
display(pd.DataFrame(settle).T.map(lambda t: f"{t[0]:+.2f} / {t[1]:.2f}"))

## Reading
* **Shadow first.** An entity that is low in the shadow is not encoded at all (D, weights or the reconstruction are not enough): retention
  cannot help. High in the shadow and low live: the retention/open-target path lost it.
* **B plain vs `sg` at D = 128.** If `sg` holds the hand too, D alone did it; B's contribution is what it holds that `sg` does not (the
  noise dropped, the position, the mob).
* **B + weights vs B plain:** prediction from the earlier runs: the weights are what lifted the agent position (0.72 to 0.84); the hand at
  0.88 % is borderline without them at D = 128.
* **Mob and arrow** are the new content: the mob is predictable now, so unlike before it should be kept, not dropped as noise. `twinkle`
  is the noise entity: B should still drop it (low live).
* Unsettled curves are provisional.